# Chapter: One-vs-One Classification Analysis

## Pairwise Discriminability in Brain Connectivity Fingerprinting

---

## Executive Rationale

### Why One-vs-One?

| Strategy | Question Answered | Granularity | Classifiers |
|----------|------------------|-------------|-------------|
| **Multinomial** | "Which region does this belong to?" | Global | 1 |
| **OvR** | "Is this Region k or not?" | Regional | K |
| **OvO** | "Is this Region A or Region B?" | Pairwise | K(K-1)/2 |

### Unique Contributions of OvO

1. **Direct Pairwise Discriminability:** How easily can we distinguish Region A from Region B specifically?
2. **Confusion Topology:** Which region pairs are functionally similar (hard to separate)?
3. **Asymmetric Confusion:** Does Region A get confused with B more than B with A?
4. **Network Boundary Sharpness:** Are within-network pairs harder to separate than cross-network pairs?

### Connection to "Error-as-Signal" Framework

$$\text{Functional Similarity}(A, B) \propto \text{Error Rate}_{A \leftrightarrow B}$$

- **High pairwise error** → Similar connectivity fingerprints → Potential functional coupling
- **Task-induced error increase** → Task causes functional convergence
- **Task-induced error decrease** → Task causes functional differentiation

---

### Hypotheses

| ID | Null Hypothesis (H₀) | Alternative (H₁) | Test |
|----|---------------------|------------------|------|
| H¹ | Multi = OvR = OvO accuracy | At least one differs | Cochran's Q + McNemar's |
| H² | Within-network = Cross-network pairwise acc | Cross-network > Within | Mann-Whitney U |
| H³ | Homologous = Non-homologous discriminability | Homologous pairs differ | Paired t-test |
| H⁴ | Pairwise discriminability same Rest/Task | Task alters patterns | Permutation test |
| H⁵ | All network pairs equally discriminable | Some pairs differ | Kruskal-Wallis |
| H⁶ | Confusion symmetric P(A→B) = P(B→A) | Asymmetric confusion exists | Binomial test |

---

### Analysis Structure

1. Setup & Data Loading
2. Three-Way Performance Comparison (H¹)
3. Pairwise Accuracy Matrix Construction
4. Network-Level Discriminability (H², H⁵)
5. Region-Level Confusion Profiles
6. Pairwise Analysis: Most/Least Confusable Pairs
7. Homologous Inter-Hemispheric Pairs (H³)
8. Task-Induced Pairwise Reorganization (H⁴)
9. Confusion Asymmetry Analysis (H⁶)
10. Integration with Error-as-Signal Framework
11. Hypothesis Testing Summary
12. Conclusions

---
## 1. Setup & Data Loading

In [22]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================

import numpy as np
import pandas as pd
import json
from pathlib import Path
import warnings
from collections import Counter
from itertools import combinations

# Machine Learning
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

# Statistics
from scipy import stats
from scipy.stats import (
    wilcoxon, ttest_rel, ttest_ind, mannwhitneyu, kruskal,
    binomtest, chi2_contingency, spearmanr, pearsonr
)
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import mcnemar, cochrans_q

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

# Colors
COLORS = {
    'multinomial': '#2b8cbe',
    'ovr': '#e6550d',
    'ovo': '#31a354',
    'rest': '#756bb1',
    'task': '#de2d26',
    'within': '#fc8d59',
    'cross': '#91bfdb'
}

print("✓ Libraries imported")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")

✓ Libraries imported
  NumPy: 2.3.4
  Pandas: 2.3.3


In [23]:
# =============================================================================
# PATH CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path('/home/sjoon/projects/brain_connectivity_classifier')
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

PATHS = {
    # Full Model
    'full_multi': RESULTS_DIR / 'full_connectivity_analysis' / 'multinomial',
    'full_multi_task': RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing',
    'full_ovr': RESULTS_DIR / 'full_connectivity_analysis' / 'ovr',
    'full_ovr_task': RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing_ovr',
    'full_ovo': RESULTS_DIR / 'full_connectivity_analysis' / 'ovo',
    'full_ovo_task': RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing_ovo',
    # Left Hemisphere
    'left_multi': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'multinomial',
    'left_multi_task': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_multinomial',
    'left_ovr': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'ovr',
    'left_ovr_task': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_ovr',
    'left_ovo': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'ovo',
    'left_ovo_task': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_ovo',
    # Right Hemisphere
    'right_multi': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'multinomial',
    'right_multi_task': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_multinomial',
    'right_ovr': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'ovr',
    'right_ovr_task': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_ovr',
    'right_ovo': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'ovo',
    'right_ovo_task': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_ovo',
}

# Verify key paths
print("Path Verification:")
key_paths = ['full_ovo', 'full_ovo_task', 'left_ovo', 'right_ovo']
for name in key_paths:
    status = "✓" if PATHS[name].exists() else "✗"
    print(f"  {status} {name}: {PATHS[name].exists()}")

Path Verification:
  ✓ full_ovo: True
  ✓ full_ovo_task: True
  ✓ left_ovo: True
  ✓ right_ovo: True


In [24]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_json(fp):
    with open(fp, 'r') as f:
        return json.load(f)

def load_npy(fp):
    return np.load(fp, allow_pickle=True)

def load_npy_optional(fp):
    """Load numpy file if exists, otherwise return None."""
    if Path(fp).exists():
        return np.load(fp, allow_pickle=True)
    return None

def load_csv(fp):
    return pd.read_csv(fp)

def safe_divide(num, denom, default=0):
    return num / denom if denom > 0 else default

class DataLoader:
    """Load model results with automatic detection of file types."""
    def __init__(self, path):
        self.path = Path(path)
    
    def load_cv(self):
        """
        Load CV results. Handles both probability-based models (Multinomial, OvR)
        and decision-function-based models (OvO).
        """
        result = {
            'predictions': load_npy(self.path / 'cv_predictions.npy'),
            'true_labels': load_npy(self.path / 'cv_true_labels.npy'),
            'confusion_matrix': load_npy(self.path / 'confusion_matrix.npy'),
            'metrics': load_json(self.path / 'overall_metrics.json')
        }
        
        # Try to load probabilities (Multinomial, OvR)
        prob_path = self.path / 'cv_probabilities.npy'
        if prob_path.exists():
            result['probabilities'] = load_npy(prob_path)
        else:
            result['probabilities'] = None
        
        # Try to load decision functions (OvO) - check both singular and plural
        for df_name in ['cv_decision_functions.npy', 'cv_decision_function.npy']:
            df_path = self.path / df_name
            if df_path.exists():
                result['decision_functions'] = load_npy(df_path)
                break
        else:
            result['decision_functions'] = None
        
        return result
    
    def load_task(self):
        """
        Load task testing results. Handles both probability-based and 
        decision-function-based models.
        """
        result = {
            'predictions': load_npy(self.path / 'task_predictions.npy'),
            'true_labels': load_npy(self.path / 'task_true_labels.npy'),
            'confusion_matrix': load_npy(self.path / 'task_confusion_matrix.npy'),
            'summary': load_json(self.path / 'task_testing_summary.json')
        }
        
        # Try to load probabilities (Multinomial, OvR)
        prob_path = self.path / 'task_probabilities.npy'
        if prob_path.exists():
            result['probabilities'] = load_npy(prob_path)
        else:
            result['probabilities'] = None
        
        # Try to load decision functions (OvO) - check both singular and plural
        for df_name in ['task_decision_functions.npy', 'task_decision_function.npy']:
            df_path = self.path / df_name
            if df_path.exists():
                result['decision_functions'] = load_npy(df_path)
                break
        else:
            result['decision_functions'] = None
        
        return result

print("✓ Helper functions defined")

✓ Helper functions defined


In [25]:
# =============================================================================
# DATA LOADING
# =============================================================================

print("="*80)
print("LOADING ALL MODEL DATA")
print("="*80)

# Full Model (232 regions)
print("\n[Full Model - 232 regions]")
full_multi_cv = DataLoader(PATHS['full_multi']).load_cv()
full_multi_task = DataLoader(PATHS['full_multi_task']).load_task()
full_ovr_cv = DataLoader(PATHS['full_ovr']).load_cv()
full_ovr_task = DataLoader(PATHS['full_ovr_task']).load_task()
full_ovo_cv = DataLoader(PATHS['full_ovo']).load_cv()
full_ovo_task = DataLoader(PATHS['full_ovo_task']).load_task()
print(f"  Multinomial: {len(full_multi_cv['predictions']):,} samples")
print(f"  OvR: {len(full_ovr_cv['predictions']):,} samples")
print(f"  OvO: {len(full_ovo_cv['predictions']):,} samples")

# Left Hemisphere (116 regions)
print("\n[Left Hemisphere - 116 regions]")
left_multi_cv = DataLoader(PATHS['left_multi']).load_cv()
left_multi_task = DataLoader(PATHS['left_multi_task']).load_task()
left_ovr_cv = DataLoader(PATHS['left_ovr']).load_cv()
left_ovr_task = DataLoader(PATHS['left_ovr_task']).load_task()
left_ovo_cv = DataLoader(PATHS['left_ovo']).load_cv()
left_ovo_task = DataLoader(PATHS['left_ovo_task']).load_task()
print(f"  OvO: {len(left_ovo_cv['predictions']):,} samples")

# Right Hemisphere (116 regions)
print("\n[Right Hemisphere - 116 regions]")
right_multi_cv = DataLoader(PATHS['right_multi']).load_cv()
right_multi_task = DataLoader(PATHS['right_multi_task']).load_task()
right_ovr_cv = DataLoader(PATHS['right_ovr']).load_cv()
right_ovr_task = DataLoader(PATHS['right_ovr_task']).load_task()
right_ovo_cv = DataLoader(PATHS['right_ovo']).load_cv()
right_ovo_task = DataLoader(PATHS['right_ovo_task']).load_task()
print(f"  OvO: {len(right_ovo_cv['predictions']):,} samples")

# Region Information
region_info = load_csv(PATHS['full_multi'] / 'region_info.csv')
print(f"\n✓ Loaded region info: {len(region_info)} regions")

LOADING ALL MODEL DATA

[Full Model - 232 regions]
  Multinomial: 51,968 samples
  OvR: 51,968 samples
  OvO: 51,968 samples

[Left Hemisphere - 116 regions]
  OvO: 25,984 samples

[Right Hemisphere - 116 regions]
  OvO: 25,984 samples

✓ Loaded region info: 232 regions


In [26]:
# =============================================================================
# NETWORK MAPPING & CONSTANTS
# =============================================================================

NETWORK_MAPPING = {
    'VisCent': 'Visual', 'VisPeri': 'Visual',
    'SomMotA': 'Somatomotor', 'SomMotB': 'Somatomotor',
    'DorsAttnA': 'Dorsal Attention', 'DorsAttnB': 'Dorsal Attention',
    'SalVentAttnA': 'Salience/Ventral Attention', 'SalVentAttnB': 'Salience/Ventral Attention',
    'LimbicA': 'Limbic', 'LimbicB': 'Limbic',
    'ContA': 'Control', 'ContB': 'Control', 'ContC': 'Control',
    'DefaultA': 'Default', 'DefaultB': 'Default', 'DefaultC': 'Default',
    'TempPar': 'Default',
    'Hippocampus_ant': 'Subcortical', 'Hippocampus_post': 'Subcortical',
    'Amygdala_lat': 'Subcortical', 'Amygdala_med': 'Subcortical',
    'Thalamus_DA': 'Subcortical', 'Thalamus_DP': 'Subcortical',
    'Thalamus_VA': 'Subcortical', 'Thalamus_VP': 'Subcortical',
    'Caudate_ant': 'Subcortical', 'Caudate_post': 'Subcortical',
    'Putamen_ant': 'Subcortical', 'Putamen_post': 'Subcortical',
    'Pallidum_ant': 'Subcortical', 'Pallidum_post': 'Subcortical',
    'Accumbens_core': 'Subcortical', 'Accumbens_shell': 'Subcortical'
}

region_info['major_network'] = region_info['network'].map(NETWORK_MAPPING)

# Hemisphere-specific
region_info_left = region_info[region_info['hemisphere'] == 'left'].reset_index(drop=True)
region_info_right = region_info[region_info['hemisphere'] == 'right'].reset_index(drop=True)

# Constants
NETWORKS = ['Visual', 'Somatomotor', 'Dorsal Attention', 'Salience/Ventral Attention',
            'Limbic', 'Control', 'Default', 'Subcortical']
N_NETWORKS = len(NETWORKS)
N_REGIONS_FULL = 232
N_REGIONS_HEMI = 116
N_PAIRS_FULL = N_REGIONS_FULL * (N_REGIONS_FULL - 1) // 2  # 26,796
N_PAIRS_HEMI = N_REGIONS_HEMI * (N_REGIONS_HEMI - 1) // 2  # 6,670

print(f"Regions: Full={N_REGIONS_FULL}, Hemisphere={N_REGIONS_HEMI}")
print(f"Pairwise classifiers: Full={N_PAIRS_FULL:,}, Hemisphere={N_PAIRS_HEMI:,}")
print(f"\nRegions per Network:")
print(region_info['major_network'].value_counts().reindex(NETWORKS))

Regions: Full=232, Hemisphere=116
Pairwise classifiers: Full=26,796, Hemisphere=6,670

Regions per Network:
major_network
Visual                        24
Somatomotor                   34
Dorsal Attention              22
Salience/Ventral Attention    26
Limbic                        14
Control                       37
Default                       43
Subcortical                   32
Name: count, dtype: int64


---
## 2. Three-Way Performance Comparison

### Comparing Multinomial vs OvR vs OvO

In [27]:
# =============================================================================
# AGGREGATE METRICS CALCULATION
# =============================================================================

def calculate_metrics(y_true, y_pred):
    """Calculate comprehensive metrics."""
    n = len(y_true)
    n_err = (y_true != y_pred).sum()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'kappa': cohen_kappa_score(y_true, y_pred),
        'n_samples': n,
        'n_errors': n_err,
        'error_rate': n_err / n
    }

# Calculate for all strategies and conditions
metrics_all = {}
for prefix in ['full', 'left', 'right']:
    for strategy in ['multi', 'ovr', 'ovo']:
        for condition in ['cv', 'task']:
            key = f"{prefix}_{strategy}_{condition}"
            data = eval(f"{prefix}_{strategy}_{condition}")
            metrics_all[key] = calculate_metrics(data['true_labels'], data['predictions'])

print("✓ Metrics calculated for all models")

✓ Metrics calculated for all models


In [28]:
# =============================================================================
# THREE-WAY PERFORMANCE TABLE
# =============================================================================

print("="*155)
print("THREE-WAY PERFORMANCE COMPARISON: Multinomial vs OvR vs OvO")
print("="*155)

print(f"\n{'Model':<15} │ {'Multi CV':>10} {'Multi Task':>12} {'Multi Δ':>10} │ "
      f"{'OvR CV':>10} {'OvR Task':>11} {'OvR Δ':>9} │ "
      f"{'OvO CV':>10} {'OvO Task':>11} {'OvO Δ':>9}")
print("-"*150)

for model, n in [('Full', 232), ('Left', 116), ('Right', 116)]:
    prefix = model.lower()
    m_cv = metrics_all[f'{prefix}_multi_cv']['accuracy']
    m_task = metrics_all[f'{prefix}_multi_task']['accuracy']
    o_cv = metrics_all[f'{prefix}_ovr_cv']['accuracy']
    o_task = metrics_all[f'{prefix}_ovr_task']['accuracy']
    v_cv = metrics_all[f'{prefix}_ovo_cv']['accuracy']
    v_task = metrics_all[f'{prefix}_ovo_task']['accuracy']
    
    print(f"{model} ({n})<15 │ {m_cv:>9.2%} {m_task:>11.2%} {m_cv-m_task:>+9.2%} │ "
          f"{o_cv:>9.2%} {o_task:>10.2%} {o_cv-o_task:>+8.2%} │ "
          f"{v_cv:>9.2%} {v_task:>10.2%} {v_cv-v_task:>+8.2%}")

print("="*155)

THREE-WAY PERFORMANCE COMPARISON: Multinomial vs OvR vs OvO

Model           │   Multi CV   Multi Task    Multi Δ │     OvR CV    OvR Task     OvR Δ │     OvO CV    OvO Task     OvO Δ
------------------------------------------------------------------------------------------------------------------------------------------------------
Full (232)<15 │    92.41%      89.24%    +3.17% │    93.17%     89.63%   +3.54% │    86.88%     83.05%   +3.83%
Left (116)<15 │    91.82%      86.93%    +4.89% │    92.26%     87.57%   +4.70% │    88.14%     83.59%   +4.56%
Right (116)<15 │    91.36%      86.31%    +5.05% │    92.08%     87.10%   +4.98% │    87.53%     82.36%   +5.17%


In [29]:
# =============================================================================
# HYPOTHESIS H¹: THREE-WAY COMPARISON (COCHRAN'S Q + McNEMAR'S)
# =============================================================================

def mcnemar_test(y_true, pred1, pred2):
    """McNemar's test for paired classifier comparison."""
    c1 = pred1 == y_true
    c2 = pred2 == y_true
    b = np.sum(~c1 & c2)  # pred1 wrong, pred2 correct
    c = np.sum(c1 & ~c2)  # pred1 correct, pred2 wrong
    
    if b + c > 0:
        stat = (abs(b - c) - 1)**2 / (b + c)
        p = 1 - stats.chi2.cdf(stat, df=1)
    else:
        stat, p = 0, 1.0
    
    return {'b': b, 'c': c, 'chi2': stat, 'p_value': p, 'better': 'First' if c > b else ('Second' if b > c else 'Equal')}

def cochrans_q_test(y_true, *predictions):
    """
    Cochran's Q test for comparing 3+ classifiers.
    """
    n = len(y_true)
    k = len(predictions)
    
    # Create binary matrix: rows=samples, cols=classifiers
    correct = np.column_stack([(p == y_true).astype(int) for p in predictions])
    
    # Row sums and column sums
    row_sums = correct.sum(axis=1)
    col_sums = correct.sum(axis=0)
    
    # Cochran's Q statistic
    N = correct.sum()
    numerator = (k - 1) * (k * np.sum(col_sums**2) - N**2)
    denominator = k * N - np.sum(row_sums**2)
    
    if denominator == 0:
        return {'Q': 0, 'p_value': 1.0, 'df': k-1}
    
    Q = numerator / denominator
    p_value = 1 - stats.chi2.cdf(Q, df=k-1)
    
    return {'Q': Q, 'p_value': p_value, 'df': k-1}

print("="*120)
print("HYPOTHESIS H¹: THREE-WAY COMPARISON")
print("="*120)
print("\nH₀: Multinomial = OvR = OvO accuracy")
print("H₁: At least one strategy differs\n")

# Cochran's Q test for Full Model CV
y_true = full_multi_cv['true_labels']
pred_m = full_multi_cv['predictions']
pred_r = full_ovr_cv['predictions']
pred_o = full_ovo_cv['predictions']

q_result = cochrans_q_test(y_true, pred_m, pred_r, pred_o)
print(f"Cochran's Q Test (Full Model, CV):")
print(f"  Q statistic: {q_result['Q']:.2f}")
print(f"  df: {q_result['df']}")
print(f"  p-value: {q_result['p_value']:.2e}")
print(f"  Conclusion: {'Reject H₀ — strategies differ' if q_result['p_value'] < 0.05 else 'Fail to reject H₀'}")

# Pairwise McNemar's tests
print("\n" + "-"*80)
print("PAIRWISE McNEMAR'S TESTS (with Bonferroni correction, α=0.05/3=0.0167):")
print("-"*80)

pairs = [
    ('Multi vs OvR', pred_m, pred_r),
    ('Multi vs OvO', pred_m, pred_o),
    ('OvR vs OvO', pred_r, pred_o)
]

print(f"{'Comparison':<18} {'b':>8} {'c':>8} {'χ²':>10} {'p-value':>12} {'Better':>12} {'Sig':>6}")
for name, p1, p2 in pairs:
    result = mcnemar_test(y_true, p1, p2)
    sig = '*' if result['p_value'] < 0.0167 else 'ns'
    print(f"{name:<18} {result['b']:>8} {result['c']:>8} {result['chi2']:>10.2f} "
          f"{result['p_value']:>12.2e} {result['better']:>12} {sig:>6}")

HYPOTHESIS H¹: THREE-WAY COMPARISON

H₀: Multinomial = OvR = OvO accuracy
H₁: At least one strategy differs

Cochran's Q Test (Full Model, CV):
  Q statistic: 3373.24
  df: 2
  p-value: 0.00e+00
  Conclusion: Reject H₀ — strategies differ

--------------------------------------------------------------------------------
PAIRWISE McNEMAR'S TESTS (with Bonferroni correction, α=0.05/3=0.0167):
--------------------------------------------------------------------------------
Comparison                b        c         χ²      p-value       Better    Sig
Multi vs OvR           1333      939      67.98     1.11e-16       Second      *
Multi vs OvO            493     3369    2140.24     0.00e+00        First      *
OvR vs OvO              966     4236    2054.28     0.00e+00        First      *


In [30]:
# =============================================================================
# ERROR OVERLAP ANALYSIS (VENN-STYLE)
# =============================================================================

def error_overlap_three(y_true, pred_m, pred_r, pred_o):
    """Analyze error overlap between three classifiers."""
    err_m = set(np.where(pred_m != y_true)[0])
    err_r = set(np.where(pred_r != y_true)[0])
    err_o = set(np.where(pred_o != y_true)[0])
    
    return {
        'multi_only': len(err_m - err_r - err_o),
        'ovr_only': len(err_r - err_m - err_o),
        'ovo_only': len(err_o - err_m - err_r),
        'multi_ovr': len((err_m & err_r) - err_o),
        'multi_ovo': len((err_m & err_o) - err_r),
        'ovr_ovo': len((err_r & err_o) - err_m),
        'all_three': len(err_m & err_r & err_o),
        'total_m': len(err_m),
        'total_r': len(err_r),
        'total_o': len(err_o)
    }

print("\n" + "="*100)
print("ERROR OVERLAP ANALYSIS")
print("="*100)

for model, prefix in [('Full (CV)', 'full'), ('Full (Task)', 'full')]:
    suffix = 'cv' if 'CV' in model else 'task'
    m_data = eval(f"{prefix}_multi_{suffix}")
    r_data = eval(f"{prefix}_ovr_{suffix}")
    o_data = eval(f"{prefix}_ovo_{suffix}")
    
    overlap = error_overlap_three(
        m_data['true_labels'], m_data['predictions'],
        r_data['predictions'], o_data['predictions']
    )
    
    print(f"\n{model}:")
    print(f"  Total errors: Multi={overlap['total_m']}, OvR={overlap['total_r']}, OvO={overlap['total_o']}")
    print(f"  Unique errors: Multi-only={overlap['multi_only']}, OvR-only={overlap['ovr_only']}, OvO-only={overlap['ovo_only']}")
    print(f"  Shared (2 methods): Multi∩OvR={overlap['multi_ovr']}, Multi∩OvO={overlap['multi_ovo']}, OvR∩OvO={overlap['ovr_ovo']}")
    print(f"  Shared (all 3): {overlap['all_three']}")


ERROR OVERLAP ANALYSIS

Full (CV):
  Total errors: Multi=3943, OvR=3549, OvO=6819
  Unique errors: Multi-only=166, OvR-only=639, OvO-only=3069
  Shared (2 methods): Multi∩OvR=327, Multi∩OvO=1167, OvR∩OvO=300
  Shared (all 3): 2283

Full (Task):
  Total errors: Multi=4993, OvR=4810, OvO=7865
  Unique errors: Multi-only=210, OvR-only=772, OvO-only=3112
  Shared (2 methods): Multi∩OvR=429, Multi∩OvO=1144, OvR∩OvO=399
  Shared (all 3): 3210


In [31]:
# =============================================================================
# THREE-WAY PERFORMANCE VISUALIZATION
# =============================================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['<b>Cross-Validation (Rest)</b>', '<b>Task Testing (Gender Stroop)</b>']
)

models = ['Full (232)', 'Left (116)', 'Right (116)']
prefixes = ['full', 'left', 'right']

for col_idx, condition in enumerate(['cv', 'task']):
    multi_acc = [metrics_all[f'{p}_multi_{condition}']['accuracy'] for p in prefixes]
    ovr_acc = [metrics_all[f'{p}_ovr_{condition}']['accuracy'] for p in prefixes]
    ovo_acc = [metrics_all[f'{p}_ovo_{condition}']['accuracy'] for p in prefixes]
    
    fig.add_trace(go.Bar(name='Multinomial' if col_idx==0 else None, x=models, y=multi_acc,
                         marker_color=COLORS['multinomial'], text=[f'{x:.1%}' for x in multi_acc],
                         textposition='outside', showlegend=(col_idx==0), legendgroup='multi'),
                  row=1, col=col_idx+1)
    fig.add_trace(go.Bar(name='OvR' if col_idx==0 else None, x=models, y=ovr_acc,
                         marker_color=COLORS['ovr'], text=[f'{x:.1%}' for x in ovr_acc],
                         textposition='outside', showlegend=(col_idx==0), legendgroup='ovr'),
                  row=1, col=col_idx+1)
    fig.add_trace(go.Bar(name='OvO' if col_idx==0 else None, x=models, y=ovo_acc,
                         marker_color=COLORS['ovo'], text=[f'{x:.1%}' for x in ovo_acc],
                         textposition='outside', showlegend=(col_idx==0), legendgroup='ovo'),
                  row=1, col=col_idx+1)

fig.update_layout(
    title=dict(text='<b>Classification Accuracy: Three-Way Comparison</b>', x=0.5),
    barmode='group', template='simple_white', height=450, width=1100,
    yaxis=dict(range=[0.85, 1.0], tickformat='.0%', title='Accuracy'),
    yaxis2=dict(range=[0.85, 1.0], tickformat='.0%', title='Accuracy'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.show()

---
## 3. Pairwise Accuracy Matrix Construction

### Computing Discriminability Between Every Region Pair

In [32]:
# =============================================================================
# PAIRWISE ACCURACY COMPUTATION
# =============================================================================

def compute_pairwise_accuracy_matrix(y_true, y_pred, n_regions):
    """
    Compute pairwise accuracy for all region pairs.
    
    For each pair (i, j), filter samples where true label is i or j,
    and compute classification accuracy on that subset.
    
    Returns:
        pairwise_acc: n_regions x n_regions symmetric matrix
        pairwise_counts: n_regions x n_regions matrix of sample counts
    """
    pairwise_acc = np.zeros((n_regions, n_regions))
    pairwise_counts = np.zeros((n_regions, n_regions))
    
    for i in range(n_regions):
        for j in range(i+1, n_regions):
            # Filter samples for this pair
            mask = (y_true == i) | (y_true == j)
            y_sub = y_true[mask]
            pred_sub = y_pred[mask]
            
            n_samples = mask.sum()
            if n_samples > 0:
                acc = accuracy_score(y_sub, pred_sub)
            else:
                acc = np.nan
            
            pairwise_acc[i, j] = acc
            pairwise_acc[j, i] = acc
            pairwise_counts[i, j] = n_samples
            pairwise_counts[j, i] = n_samples
    
    # Diagonal = 1 (self-accuracy)
    np.fill_diagonal(pairwise_acc, 1.0)
    
    return pairwise_acc, pairwise_counts

print("Computing pairwise accuracy matrices...")
print("  (This may take a moment for 26,796 pairs)")

# Full Model
pw_full_ovo_cv, pw_counts_full_cv = compute_pairwise_accuracy_matrix(
    full_ovo_cv['true_labels'], full_ovo_cv['predictions'], N_REGIONS_FULL)
pw_full_ovo_task, pw_counts_full_task = compute_pairwise_accuracy_matrix(
    full_ovo_task['true_labels'], full_ovo_task['predictions'], N_REGIONS_FULL)

# Left Hemisphere
pw_left_ovo_cv, _ = compute_pairwise_accuracy_matrix(
    left_ovo_cv['true_labels'], left_ovo_cv['predictions'], N_REGIONS_HEMI)
pw_left_ovo_task, _ = compute_pairwise_accuracy_matrix(
    left_ovo_task['true_labels'], left_ovo_task['predictions'], N_REGIONS_HEMI)

# Right Hemisphere
pw_right_ovo_cv, _ = compute_pairwise_accuracy_matrix(
    right_ovo_cv['true_labels'], right_ovo_cv['predictions'], N_REGIONS_HEMI)
pw_right_ovo_task, _ = compute_pairwise_accuracy_matrix(
    right_ovo_task['true_labels'], right_ovo_task['predictions'], N_REGIONS_HEMI)

print("✓ Pairwise accuracy matrices computed")
print(f"  Full model shape: {pw_full_ovo_cv.shape}")
print(f"  Mean pairwise accuracy (Full CV): {np.nanmean(pw_full_ovo_cv[np.triu_indices(N_REGIONS_FULL, k=1)]):.4f}")

Computing pairwise accuracy matrices...
  (This may take a moment for 26,796 pairs)
✓ Pairwise accuracy matrices computed
  Full model shape: (232, 232)
  Mean pairwise accuracy (Full CV): 0.8688


In [33]:
# =============================================================================
# PAIRWISE ACCURACY SUMMARY STATISTICS
# =============================================================================

def pairwise_stats(pw_matrix, name):
    """Compute statistics on upper triangle (unique pairs)."""
    upper = pw_matrix[np.triu_indices(pw_matrix.shape[0], k=1)]
    upper = upper[~np.isnan(upper)]
    
    return {
        'name': name,
        'n_pairs': len(upper),
        'mean': np.mean(upper),
        'std': np.std(upper),
        'min': np.min(upper),
        'max': np.max(upper),
        'q25': np.percentile(upper, 25),
        'median': np.median(upper),
        'q75': np.percentile(upper, 75),
        'below_90': (upper < 0.90).sum(),
        'below_80': (upper < 0.80).sum()
    }

print("="*120)
print("PAIRWISE ACCURACY STATISTICS")
print("="*120)

stats_list = [
    pairwise_stats(pw_full_ovo_cv, 'Full CV'),
    pairwise_stats(pw_full_ovo_task, 'Full Task'),
    pairwise_stats(pw_left_ovo_cv, 'Left CV'),
    pairwise_stats(pw_left_ovo_task, 'Left Task'),
    pairwise_stats(pw_right_ovo_cv, 'Right CV'),
    pairwise_stats(pw_right_ovo_task, 'Right Task'),
]

print(f"\n{'Model':<12} {'Pairs':>8} {'Mean':>8} {'Std':>8} {'Min':>8} {'Median':>8} {'Max':>8} {'<90%':>8} {'<80%':>8}")
print("-"*90)

for s in stats_list:
    print(f"{s['name']:<12} {s['n_pairs']:>8,} {s['mean']:>7.2%} {s['std']:>7.2%} {s['min']:>7.2%} "
          f"{s['median']:>7.2%} {s['max']:>7.2%} {s['below_90']:>8,} {s['below_80']:>8,}")

PAIRWISE ACCURACY STATISTICS

Model           Pairs     Mean      Std      Min   Median      Max     <90%     <80%
------------------------------------------------------------------------------------------
Full CV        26,796  86.88%   5.87%  43.75%  88.17%  98.21%   18,576    2,682
Full Task      26,796  83.05%   8.05%  24.50%  84.50%  98.00%   21,758    7,446
Left CV         6,670  88.14%   6.03%  57.37%  89.73%  97.99%    3,478      740
Left Task       6,670  83.59%   8.48%  39.50%  85.00%  98.25%    4,975    1,855
Right CV        6,670  87.53%   5.87%  55.58%  89.06%  97.54%    3,985      770
Right Task      6,670  82.36%   8.77%  37.75%  84.50%  99.00%    5,445    2,117


In [34]:
# =============================================================================
# PAIRWISE ACCURACY DISTRIBUTION
# =============================================================================

upper_cv = pw_full_ovo_cv[np.triu_indices(N_REGIONS_FULL, k=1)]
upper_task = pw_full_ovo_task[np.triu_indices(N_REGIONS_FULL, k=1)]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['<b>Pairwise Accuracy Distribution</b>', '<b>CV vs Task Comparison</b>']
)

# Distribution
fig.add_trace(go.Histogram(x=upper_cv, name='CV (Rest)', opacity=0.7,
                           marker_color=COLORS['rest'], nbinsx=50), row=1, col=1)
fig.add_trace(go.Histogram(x=upper_task, name='Task', opacity=0.7,
                           marker_color=COLORS['task'], nbinsx=50), row=1, col=1)

# Scatter CV vs Task
fig.add_trace(go.Scatter(x=upper_cv, y=upper_task, mode='markers',
                         marker=dict(size=3, opacity=0.3, color=COLORS['ovo']),
                         name='Pairs', showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=[0.5, 1], y=[0.5, 1], mode='lines',
                         line=dict(dash='dash', color='gray'), showlegend=False), row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Pairwise Accuracy: Full Model OvO</b>', x=0.5),
    barmode='overlay', template='simple_white', height=400, width=1000,
    xaxis_title='Pairwise Accuracy', yaxis_title='Count',
    xaxis2_title='CV Accuracy', yaxis2_title='Task Accuracy'
)
fig.show()

---
## 4. Network-Level Discriminability Analysis

### Within-Network vs Cross-Network Pairwise Accuracy

In [35]:
# =============================================================================
# NETWORK-PAIR DISCRIMINABILITY MATRIX
# =============================================================================

def compute_network_pair_discriminability(pw_matrix, region_info, networks):
    """
    Compute mean pairwise accuracy for each network pair.
    
    Returns 8x8 matrix where D[i,j] = mean pairwise accuracy for
    all pairs where one region is in network i and one in network j.
    """
    n_networks = len(networks)
    D = np.zeros((n_networks, n_networks))
    D_count = np.zeros((n_networks, n_networks))
    
    # Map region idx to network
    region_nets = region_info['major_network'].values
    net_to_idx = {n: i for i, n in enumerate(networks)}
    
    n_regions = pw_matrix.shape[0]
    for i in range(n_regions):
        for j in range(i+1, n_regions):
            net_i = net_to_idx.get(region_nets[i])
            net_j = net_to_idx.get(region_nets[j])
            
            if net_i is not None and net_j is not None:
                acc = pw_matrix[i, j]
                if not np.isnan(acc):
                    D[net_i, net_j] += acc
                    D[net_j, net_i] += acc
                    D_count[net_i, net_j] += 1
                    D_count[net_j, net_i] += 1
    
    # Average
    D_count[D_count == 0] = 1  # Avoid division by zero
    D = D / D_count
    
    return D, D_count

# Compute for Full Model
net_D_cv, net_count_cv = compute_network_pair_discriminability(pw_full_ovo_cv, region_info, NETWORKS)
net_D_task, net_count_task = compute_network_pair_discriminability(pw_full_ovo_task, region_info, NETWORKS)
net_D_diff = net_D_cv - net_D_task  # Positive = more discriminable at rest

print("✓ Network-pair discriminability matrices computed")

✓ Network-pair discriminability matrices computed


In [36]:
# =============================================================================
# NETWORK-PAIR DISCRIMINABILITY TABLE
# =============================================================================

print("="*120)
print("NETWORK-PAIR DISCRIMINABILITY (Mean Pairwise Accuracy)")
print("="*120)

# Create DataFrame for display
net_D_df = pd.DataFrame(net_D_cv, index=NETWORKS, columns=NETWORKS)
print("\nCV (Rest) - Network × Network Mean Pairwise Accuracy:")
print(net_D_df.round(3).to_string())

# Within vs Cross comparison
within_acc = []
cross_acc = []

for i in range(N_NETWORKS):
    within_acc.append(net_D_cv[i, i])
    for j in range(i+1, N_NETWORKS):
        cross_acc.append(net_D_cv[i, j])

print(f"\nWithin-Network Mean: {np.mean(within_acc):.4f} (n={len(within_acc)})")
print(f"Cross-Network Mean: {np.mean(cross_acc):.4f} (n={len(cross_acc)})")

NETWORK-PAIR DISCRIMINABILITY (Mean Pairwise Accuracy)

CV (Rest) - Network × Network Mean Pairwise Accuracy:
                            Visual  Somatomotor  Dorsal Attention  Salience/Ventral Attention  Limbic  Control  Default  Subcortical
Visual                       0.938        0.922             0.912                       0.902   0.862    0.912    0.907        0.857
Somatomotor                  0.922        0.906             0.896                       0.886   0.846    0.896    0.891        0.841
Dorsal Attention             0.912        0.896             0.887                       0.876   0.836    0.886    0.881        0.831
Salience/Ventral Attention   0.902        0.886             0.876                       0.865   0.825    0.875    0.870        0.820
Limbic                       0.862        0.846             0.836                       0.825   0.786    0.836    0.831        0.781
Control                      0.912        0.896             0.886                       0.87

In [37]:
# =============================================================================
# HYPOTHESIS H²: WITHIN vs CROSS-NETWORK
# =============================================================================

def classify_pairs_by_network(pw_matrix, region_info, networks):
    """
    Classify all pairs as within-network or cross-network.
    Returns arrays of pairwise accuracies for each category.
    """
    region_nets = region_info['major_network'].values
    n_regions = pw_matrix.shape[0]
    
    within = []
    cross_same_hemi = []
    cross_diff_hemi = []
    
    region_hemis = region_info['hemisphere'].values
    
    for i in range(n_regions):
        for j in range(i+1, n_regions):
            acc = pw_matrix[i, j]
            if np.isnan(acc):
                continue
            
            same_net = region_nets[i] == region_nets[j]
            same_hemi = region_hemis[i] == region_hemis[j]
            
            if same_net and same_hemi:
                within.append(acc)
            elif not same_net and same_hemi:
                cross_same_hemi.append(acc)
            elif same_net and not same_hemi:
                cross_diff_hemi.append(acc)  # Homologous pairs
            else:  # Different network, different hemisphere
                cross_diff_hemi.append(acc)
    
    return {
        'within': np.array(within),
        'cross_same_hemi': np.array(cross_same_hemi),
        'cross_diff_hemi': np.array(cross_diff_hemi)
    }

pair_cats_cv = classify_pairs_by_network(pw_full_ovo_cv, region_info, NETWORKS)
pair_cats_task = classify_pairs_by_network(pw_full_ovo_task, region_info, NETWORKS)

print("="*100)
print("HYPOTHESIS H²: WITHIN-NETWORK vs CROSS-NETWORK DISCRIMINABILITY")
print("="*100)
print("\nH₀: Within-network pairwise accuracy = Cross-network pairwise accuracy")
print("H₁: Cross-network pairs are MORE discriminable (higher accuracy)\n")

within = pair_cats_cv['within']
cross = np.concatenate([pair_cats_cv['cross_same_hemi'], pair_cats_cv['cross_diff_hemi']])

print(f"Within-Network pairs: n={len(within):,}, Mean={np.mean(within):.4f}, SD={np.std(within):.4f}")
print(f"Cross-Network pairs:  n={len(cross):,}, Mean={np.mean(cross):.4f}, SD={np.std(cross):.4f}")
print(f"Difference (Cross - Within): {np.mean(cross) - np.mean(within):+.4f}")

# Mann-Whitney U test (one-tailed: cross > within)
u_stat, p_two = mannwhitneyu(cross, within, alternative='two-sided')
# For one-tailed, check if cross > within
p_one = p_two / 2 if np.mean(cross) > np.mean(within) else 1 - p_two / 2

# Effect size (rank-biserial)
r = 1 - (2 * u_stat) / (len(cross) * len(within))

print(f"\nMann-Whitney U: {u_stat:,.0f}")
print(f"p-value (one-tailed): {p_one:.2e}")
print(f"Effect size (r): {r:.4f}")
print(f"\nConclusion: {'Reject H₀ — Cross-network pairs MORE discriminable' if p_one < 0.05 else 'Fail to reject H₀'}")

HYPOTHESIS H²: WITHIN-NETWORK vs CROSS-NETWORK DISCRIMINABILITY

H₀: Within-network pairwise accuracy = Cross-network pairwise accuracy
H₁: Cross-network pairs are MORE discriminable (higher accuracy)

Within-Network pairs: n=1,743, Mean=0.8711, SD=0.0642
Cross-Network pairs:  n=25,053, Mean=0.8686, SD=0.0583
Difference (Cross - Within): -0.0025

Mann-Whitney U: 20,707,576
p-value (one-tailed): 1.00e+00
Effect size (r): 0.0516

Conclusion: Fail to reject H₀


In [38]:
# =============================================================================
# NETWORK-PAIR HEATMAPS (3-PANEL: CV, TASK, DIFFERENCE)
# =============================================================================

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['<b>CV (Rest)</b>', '<b>Task</b>', '<b>Δ (CV - Task)</b>'],
    horizontal_spacing=0.08
)

seq_cs = [[0, '#f7fcf5'], [0.5, '#74c476'], [1, '#00441b']]
div_cs = [[0, '#2166ac'], [0.5, '#f7f7f7'], [1, '#b2182b']]

# CV
fig.add_trace(go.Heatmap(z=net_D_cv, x=NETWORKS, y=NETWORKS, colorscale=seq_cs,
                         zmin=0.9, zmax=1.0, showscale=False,
                         hovertemplate='%{y} × %{x}: %{z:.3f}<extra></extra>'), row=1, col=1)

# Task
fig.add_trace(go.Heatmap(z=net_D_task, x=NETWORKS, y=NETWORKS, colorscale=seq_cs,
                         zmin=0.9, zmax=1.0, showscale=True,
                         colorbar=dict(title='Accuracy', x=0.64, len=0.8),
                         hovertemplate='%{y} × %{x}: %{z:.3f}<extra></extra>'), row=1, col=2)

# Difference
limit = np.max(np.abs(net_D_diff))
fig.add_trace(go.Heatmap(z=net_D_diff, x=NETWORKS, y=NETWORKS, colorscale=div_cs,
                         zmin=-limit, zmax=limit, showscale=True,
                         colorbar=dict(title='Δ', x=1.02, len=0.8),
                         hovertemplate='%{y} × %{x}: %{z:+.3f}<extra></extra>'), row=1, col=3)

fig.update_layout(
    title=dict(text='<b>Network-Pair Discriminability (OvO)</b>', x=0.5),
    template='plotly_white', height=450, width=1200
)
fig.update_xaxes(tickangle=45, tickfont=dict(size=8))
fig.update_yaxes(autorange='reversed', tickfont=dict(size=8))
fig.show()

In [39]:
# =============================================================================
# WITHIN vs CROSS-NETWORK BOX PLOT
# =============================================================================

# Build dataframe for plotting
plot_data = []
for cat, arr in pair_cats_cv.items():
    for val in arr:
        plot_data.append({'Category': cat.replace('_', ' ').title(), 'Accuracy': val, 'Condition': 'CV'})
for cat, arr in pair_cats_task.items():
    for val in arr:
        plot_data.append({'Category': cat.replace('_', ' ').title(), 'Accuracy': val, 'Condition': 'Task'})

plot_df = pd.DataFrame(plot_data)

fig = px.box(
    plot_df, x='Category', y='Accuracy', color='Condition',
    color_discrete_map={'CV': COLORS['rest'], 'Task': COLORS['task']},
    title='<b>Pairwise Accuracy by Network Relationship</b>'
)

fig.add_hline(y=0.95, line_dash='dash', line_color='gray', annotation_text='95%')

fig.update_layout(
    template='simple_white', height=500, width=900,
    yaxis_title='Pairwise Accuracy', xaxis_title=''
)
fig.show()

In [40]:
# =============================================================================
# HYPOTHESIS H⁵: NETWORK-PAIR DIFFERENTIAL DISCRIMINABILITY
# =============================================================================

print("="*100)
print("HYPOTHESIS H⁵: NETWORK-PAIR DIFFERENTIAL DISCRIMINABILITY")
print("="*100)
print("\nH₀: All network pairs are equally discriminable")
print("H₁: Some network pairs differ in discriminability\n")

# Get pairwise accuracy for each network pair category
def get_network_pair_accuracies(pw_matrix, region_info):
    """Get all pairwise accuracies grouped by network pair."""
    region_nets = region_info['major_network'].values
    n_regions = pw_matrix.shape[0]
    
    pair_accs = {}
    
    for i in range(n_regions):
        for j in range(i+1, n_regions):
            acc = pw_matrix[i, j]
            if np.isnan(acc):
                continue
            
            net_i, net_j = region_nets[i], region_nets[j]
            key = tuple(sorted([net_i, net_j]))
            
            if key not in pair_accs:
                pair_accs[key] = []
            pair_accs[key].append(acc)
    
    return pair_accs

net_pair_accs = get_network_pair_accuracies(pw_full_ovo_cv, region_info)

# Kruskal-Wallis test
groups = list(net_pair_accs.values())
h_stat, h_p = kruskal(*groups)

# Effect size
n = sum(len(g) for g in groups)
k = len(groups)
eta_sq = (h_stat - k + 1) / (n - k) if n > k else 0

print(f"Kruskal-Wallis H: {h_stat:.2f}")
print(f"p-value: {h_p:.2e}")
print(f"Effect size (η²): {eta_sq:.4f}")
print(f"\nConclusion: {'Reject H₀ — Network pairs differ' if h_p < 0.05 else 'Fail to reject H₀'}")

# Show network pair rankings
print("\n" + "-"*70)
print("NETWORK PAIR RANKING (by mean pairwise accuracy):")
print("-"*70)

net_pair_stats = []
for (n1, n2), accs in net_pair_accs.items():
    net_pair_stats.append({
        'Pair': f"{n1[:8]}-{n2[:8]}",
        'Full': f"{n1} × {n2}",
        'Mean': np.mean(accs),
        'Std': np.std(accs),
        'N': len(accs)
    })

net_pair_df = pd.DataFrame(net_pair_stats).sort_values('Mean')
print(f"\n{'Pair':<45} {'Mean':>8} {'Std':>8} {'N':>6}")
for _, row in net_pair_df.head(10).iterrows():
    print(f"{row['Full']:<45} {row['Mean']:>7.3f} {row['Std']:>7.3f} {row['N']:>6}")
print("...")
for _, row in net_pair_df.tail(5).iterrows():
    print(f"{row['Full']:<45} {row['Mean']:>7.3f} {row['Std']:>7.3f} {row['N']:>6}")

HYPOTHESIS H⁵: NETWORK-PAIR DIFFERENTIAL DISCRIMINABILITY

H₀: All network pairs are equally discriminable
H₁: Some network pairs differ in discriminability

Kruskal-Wallis H: 10028.53
p-value: 0.00e+00
Effect size (η²): 0.3735

Conclusion: Reject H₀ — Network pairs differ

----------------------------------------------------------------------
NETWORK PAIR RANKING (by mean pairwise accuracy):
----------------------------------------------------------------------

Pair                                              Mean      Std      N
Subcortical × Subcortical                       0.775   0.094    496
Limbic × Subcortical                            0.781   0.078    448
Limbic × Limbic                                 0.786   0.053     91
Salience/Ventral Attention × Subcortical        0.820   0.072    832
Limbic × Salience/Ventral Attention             0.825   0.046    364
Default × Subcortical                           0.826   0.074   1376
Control × Subcortical                          

---
## 5. Region-Level Confusion Profiles

In [41]:
# =============================================================================
# PER-REGION CONFUSION METRICS
# =============================================================================

def compute_region_confusion_profile(pw_matrix, region_info):
    """
    For each region, compute confusion profile metrics.
    """
    n_regions = pw_matrix.shape[0]
    profiles = []
    
    for k in range(n_regions):
        # Get all pairwise accuracies for this region
        row = pw_matrix[k, :].copy()
        row[k] = np.nan  # Exclude self
        valid = row[~np.isnan(row)]
        
        if len(valid) == 0:
            continue
        
        # Find max confuser (lowest accuracy)
        min_idx = np.nanargmin(row)
        max_idx = np.nanargmax(row)
        
        profiles.append({
            'region_id': k,
            'region_name': region_info.iloc[k]['region_name'],
            'major_network': region_info.iloc[k]['major_network'],
            'hemisphere': region_info.iloc[k]['hemisphere'],
            'mean_pairwise_acc': np.mean(valid),
            'std_pairwise_acc': np.std(valid),
            'min_pairwise_acc': np.min(valid),
            'max_pairwise_acc': np.max(valid),
            'confusion_degree': (valid < 0.90).sum(),  # Pairs with <90% accuracy
            'max_confuser_id': min_idx,
            'max_confuser_name': region_info.iloc[min_idx]['region_name'],
            'max_confuser_acc': row[min_idx],
            'least_confuser_id': max_idx,
            'least_confuser_name': region_info.iloc[max_idx]['region_name'],
            'least_confuser_acc': row[max_idx]
        })
    
    return pd.DataFrame(profiles)

region_profiles_cv = compute_region_confusion_profile(pw_full_ovo_cv, region_info)
region_profiles_task = compute_region_confusion_profile(pw_full_ovo_task, region_info)

print("✓ Region confusion profiles computed")

✓ Region confusion profiles computed


In [42]:
# =============================================================================
# REGION DISCRIMINABILITY RANKING
# =============================================================================

print("="*120)
print("REGION DISCRIMINABILITY RANKING (by Mean Pairwise Accuracy)")
print("="*120)

print("\n" + "-"*100)
print("TOP 20 MOST DISCRIMINABLE REGIONS (highest mean pairwise accuracy):")
print("-"*100)
top_20 = region_profiles_cv.nlargest(20, 'mean_pairwise_acc')
print(f"{'Region':<45} {'Network':<18} {'Mean Acc':>10} {'Conf Deg':>10}")
for _, row in top_20.iterrows():
    print(f"{row['region_name']:<45} {row['major_network']:<18} {row['mean_pairwise_acc']:>9.3f} {row['confusion_degree']:>10}")

print("\n" + "-"*100)
print("BOTTOM 20 LEAST DISCRIMINABLE REGIONS (lowest mean pairwise accuracy):")
print("-"*100)
bottom_20 = region_profiles_cv.nsmallest(20, 'mean_pairwise_acc')
for _, row in bottom_20.iterrows():
    print(f"{row['region_name']:<45} {row['major_network']:<18} {row['mean_pairwise_acc']:>9.3f} {row['confusion_degree']:>10}")

REGION DISCRIMINABILITY RANKING (by Mean Pairwise Accuracy)

----------------------------------------------------------------------------------------------------
TOP 20 MOST DISCRIMINABLE REGIONS (highest mean pairwise accuracy):
----------------------------------------------------------------------------------------------------
Region                                        Network              Mean Acc   Conf Deg
LH_VisPeri_StriCal_1                          Visual                 0.927         34
RH_DefaultC_Rsp_1                             Default                0.923         39
LH_VisPeri_ExStrSup_2                         Visual                 0.921         44
RH_VisPeri_ExStrInf_1                         Visual                 0.919         50
RH_VisPeri_ExStrSup_2                         Visual                 0.919         50
LH_VisCent_ExStr_1                            Visual                 0.916         50
LH_VisCent_ExStr_5                            Visual              

In [43]:
# =============================================================================
# CONFUSION DEGREE BY NETWORK
# =============================================================================

fig = px.box(
    region_profiles_cv, x='major_network', y='mean_pairwise_acc',
    color='major_network', points='all',
    hover_data=['region_name', 'confusion_degree'],
    title='<b>Region Mean Pairwise Accuracy by Network (OvO CV)</b>'
)

fig.add_hline(y=0.95, line_dash='dash', line_color='green', annotation_text='95%')
fig.add_hline(y=0.90, line_dash='dot', line_color='orange', annotation_text='90%')

fig.update_layout(
    template='simple_white', height=500, width=1000,
    xaxis_title='Network', yaxis_title='Mean Pairwise Accuracy',
    showlegend=False
)
fig.update_xaxes(tickangle=45)
fig.show()

---
## 6. Pairwise Analysis: Most/Least Confusable Pairs

In [44]:
# =============================================================================
# EXTRACT ALL PAIRS WITH METADATA
# =============================================================================

def extract_all_pairs(pw_matrix, region_info):
    """Extract all pairwise accuracies with full metadata."""
    n_regions = pw_matrix.shape[0]
    pairs = []
    
    for i in range(n_regions):
        for j in range(i+1, n_regions):
            acc = pw_matrix[i, j]
            if np.isnan(acc):
                continue
            
            ri = region_info.iloc[i]
            rj = region_info.iloc[j]
            
            same_net = ri['major_network'] == rj['major_network']
            same_hemi = ri['hemisphere'] == rj['hemisphere']
            
            pairs.append({
                'region_i': i,
                'region_j': j,
                'name_i': ri['region_name'],
                'name_j': rj['region_name'],
                'network_i': ri['major_network'],
                'network_j': rj['major_network'],
                'hemi_i': ri['hemisphere'],
                'hemi_j': rj['hemisphere'],
                'accuracy': acc,
                'same_network': same_net,
                'same_hemisphere': same_hemi,
                'pair_type': 'Within' if same_net else 'Cross'
            })
    
    return pd.DataFrame(pairs)

all_pairs_cv = extract_all_pairs(pw_full_ovo_cv, region_info)
all_pairs_task = extract_all_pairs(pw_full_ovo_task, region_info)

print(f"Total pairs: {len(all_pairs_cv):,}")

Total pairs: 26,796


In [45]:
# =============================================================================
# MOST CONFUSABLE PAIRS
# =============================================================================

print("="*130)
print("TOP 30 MOST CONFUSABLE PAIRS (lowest pairwise accuracy)")
print("="*130)

most_confusable = all_pairs_cv.nsmallest(30, 'accuracy')
print(f"\n{'Rank':>4} {'Region A':<35} {'Region B':<35} {'Acc':>7} {'Same Net?':>10}")
print("-"*100)

for rank, (_, row) in enumerate(most_confusable.iterrows(), 1):
    same = '✓' if row['same_network'] else '✗'
    print(f"{rank:>4} {row['name_i']:<35} {row['name_j']:<35} {row['accuracy']:>6.1%} {same:>10}")

TOP 30 MOST CONFUSABLE PAIRS (lowest pairwise accuracy)

Rank Region A                            Region B                                Acc  Same Net?
----------------------------------------------------------------------------------------------------
   1 pGP-rh                              aGP-lh                               43.8%          ✓
   2 aGP-rh                              aGP-lh                               45.3%          ✓
   3 pGP-lh                              aGP-lh                               46.4%          ✓
   4 pGP-rh                              aGP-rh                               47.1%          ✓
   5 pGP-rh                              pGP-lh                               48.2%          ✓
   6 aGP-rh                              pGP-lh                               49.8%          ✓
   7 LH_LimbicA_TempPole_4               aGP-lh                               50.9%          ✗
   8 RH_DefaultA_PFCm_2                  aGP-lh                               52.

In [46]:
# =============================================================================
# MOST DISCRIMINABLE PAIRS
# =============================================================================

print("\n" + "="*130)
print("TOP 30 MOST DISCRIMINABLE PAIRS (highest pairwise accuracy)")
print("="*130)

most_discriminable = all_pairs_cv.nlargest(30, 'accuracy')
print(f"\n{'Rank':>4} {'Region A':<35} {'Region B':<35} {'Acc':>7} {'Same Net?':>10}")
print("-"*100)

for rank, (_, row) in enumerate(most_discriminable.iterrows(), 1):
    same = '✓' if row['same_network'] else '✗'
    print(f"{rank:>4} {row['name_i']:<35} {row['name_j']:<35} {row['accuracy']:>6.1%} {same:>10}")


TOP 30 MOST DISCRIMINABLE PAIRS (highest pairwise accuracy)

Rank Region A                            Region B                                Acc  Same Net?
----------------------------------------------------------------------------------------------------
   1 LH_VisPeri_StriCal_1                RH_DefaultC_Rsp_1                    98.2%          ✗
   2 LH_VisPeri_StriCal_1                LH_VisPeri_ExStrSup_2                98.0%          ✓
   3 LH_VisPeri_StriCal_1                RH_VisPeri_ExStrInf_1                97.8%          ✓
   4 LH_VisPeri_StriCal_1                RH_VisPeri_ExStrSup_2                97.8%          ✓
   5 LH_VisCent_ExStr_1                  LH_VisPeri_StriCal_1                 97.5%          ✓
   6 LH_VisCent_ExStr_5                  LH_VisPeri_StriCal_1                 97.5%          ✓
   7 LH_VisPeri_StriCal_1                RH_VisPeri_StriCal_1                 97.5%          ✓
   8 LH_VisPeri_StriCal_1                RH_ContC_pCun_1                    

---
## 7. Homologous Inter-Hemispheric Pairs (H³)

In [47]:
# =============================================================================
# IDENTIFY HOMOLOGOUS PAIRS
# =============================================================================

def identify_homologous_pairs(region_info):
    """
    Identify homologous pairs (same region in left and right hemisphere).
    """
    left_regions = region_info[region_info['hemisphere'] == 'left'].reset_index()
    right_regions = region_info[region_info['hemisphere'] == 'right'].reset_index()
    
    homologous = []
    
    for _, left in left_regions.iterrows():
        # Find matching right region (same network and similar name)
        left_name_base = left['region_name'].replace('LH_', '').replace('lh_', '').replace('Left_', '').replace('left_', '')
        
        for _, right in right_regions.iterrows():
            right_name_base = right['region_name'].replace('RH_', '').replace('rh_', '').replace('Right_', '').replace('right_', '')
            
            # Check if same network and similar naming pattern
            if (left['network'] == right['network'] and 
                left_name_base == right_name_base):
                homologous.append({
                    'left_idx': left['region_idx'],
                    'right_idx': right['region_idx'],
                    'left_name': left['region_name'],
                    'right_name': right['region_name'],
                    'network': left['major_network']
                })
                break
    
    return pd.DataFrame(homologous)

homologous_pairs = identify_homologous_pairs(region_info)
print(f"Identified {len(homologous_pairs)} homologous pairs")

# Add pairwise accuracies
homologous_pairs['acc_cv'] = homologous_pairs.apply(
    lambda r: pw_full_ovo_cv[r['left_idx'], r['right_idx']], axis=1)
homologous_pairs['acc_task'] = homologous_pairs.apply(
    lambda r: pw_full_ovo_task[r['left_idx'], r['right_idx']], axis=1)
homologous_pairs['acc_diff'] = homologous_pairs['acc_cv'] - homologous_pairs['acc_task']

print(f"\nHomologous pair accuracy (CV): Mean={homologous_pairs['acc_cv'].mean():.4f}")

Identified 78 homologous pairs

Homologous pair accuracy (CV): Mean=0.8870


In [48]:
# =============================================================================
# HYPOTHESIS H³: HOMOLOGOUS vs NON-HOMOLOGOUS
# =============================================================================

print("="*100)
print("HYPOTHESIS H³: HOMOLOGOUS PAIR DISCRIMINABILITY")
print("="*100)
print("\nH₀: Homologous pairs have same discriminability as non-homologous cross-hemisphere pairs")
print("H₁: Homologous pairs show distinct (likely lower) discriminability\n")

# Get non-homologous cross-hemisphere pairs
cross_hemi_pairs = all_pairs_cv[~all_pairs_cv['same_hemisphere']].copy()

# Mark homologous
homo_set = set()
for _, row in homologous_pairs.iterrows():
    homo_set.add((row['left_idx'], row['right_idx']))
    homo_set.add((row['right_idx'], row['left_idx']))

cross_hemi_pairs['is_homologous'] = cross_hemi_pairs.apply(
    lambda r: (r['region_i'], r['region_j']) in homo_set, axis=1)

homo_acc = cross_hemi_pairs[cross_hemi_pairs['is_homologous']]['accuracy'].values
non_homo_acc = cross_hemi_pairs[~cross_hemi_pairs['is_homologous']]['accuracy'].values

print(f"Homologous pairs: n={len(homo_acc)}, Mean={np.mean(homo_acc):.4f}")
print(f"Non-homologous cross-hemi: n={len(non_homo_acc)}, Mean={np.mean(non_homo_acc):.4f}")
print(f"Difference: {np.mean(homo_acc) - np.mean(non_homo_acc):+.4f}")

# Mann-Whitney U test
u_stat, u_p = mannwhitneyu(homo_acc, non_homo_acc, alternative='two-sided')
r_effect = 1 - (2 * u_stat) / (len(homo_acc) * len(non_homo_acc))

print(f"\nMann-Whitney U: {u_stat:,.0f}")
print(f"p-value: {u_p:.2e}")
print(f"Effect size (r): {r_effect:.4f}")
print(f"\nConclusion: {'Reject H₀ — Homologous pairs differ' if u_p < 0.05 else 'Fail to reject H₀'}")

HYPOTHESIS H³: HOMOLOGOUS PAIR DISCRIMINABILITY

H₀: Homologous pairs have same discriminability as non-homologous cross-hemisphere pairs
H₁: Homologous pairs show distinct (likely lower) discriminability

Homologous pairs: n=78, Mean=0.8870
Non-homologous cross-hemi: n=13378, Mean=0.8687
Difference: +0.0184

Mann-Whitney U: 628,608
p-value: 1.78e-03
Effect size (r): -0.2048

Conclusion: Reject H₀ — Homologous pairs differ


In [49]:
# =============================================================================
# HOMOLOGOUS PAIR ANALYSIS BY NETWORK
# =============================================================================

print("\n" + "-"*80)
print("HOMOLOGOUS PAIR ACCURACY BY NETWORK:")
print("-"*80)

homo_by_net = homologous_pairs.groupby('network').agg({
    'acc_cv': ['mean', 'std', 'count'],
    'acc_task': 'mean',
    'acc_diff': 'mean'
}).round(4)
homo_by_net.columns = ['CV Mean', 'CV Std', 'N', 'Task Mean', 'Δ']
homo_by_net = homo_by_net.sort_values('CV Mean')

print(f"\n{'Network':<28} {'N':>4} {'CV Mean':>10} {'Task Mean':>11} {'Δ':>10}")
for net, row in homo_by_net.iterrows():
    print(f"{net:<28} {int(row['N']):>4} {row['CV Mean']:>9.3f} {row['Task Mean']:>10.3f} {row['Δ']:>+9.3f}")


--------------------------------------------------------------------------------
HOMOLOGOUS PAIR ACCURACY BY NETWORK:
--------------------------------------------------------------------------------

Network                         N    CV Mean   Task Mean          Δ
Limbic                          6     0.772      0.780    -0.008
Salience/Ventral Attention     10     0.866      0.840    +0.027
Default                        14     0.881      0.850    +0.032
Dorsal Attention               10     0.885      0.883    +0.001
Control                        13     0.894      0.849    +0.045
Somatomotor                    14     0.907      0.875    +0.032
Visual                         11     0.944      0.910    +0.034


---
## 8. Task-Induced Pairwise Reorganization (H⁴)

In [50]:
# =============================================================================
# PAIRWISE TASK EFFECT
# =============================================================================

# Merge CV and Task pair data
all_pairs_cv['acc_task'] = all_pairs_task['accuracy'].values
all_pairs_cv['acc_diff'] = all_pairs_cv['accuracy'] - all_pairs_cv['acc_task']

# Rename for clarity
all_pairs_cv = all_pairs_cv.rename(columns={'accuracy': 'acc_cv'})

print("="*100)
print("TASK-INDUCED PAIRWISE CHANGES")
print("="*100)

print(f"\nOverall pairwise change (CV - Task):")
print(f"  Mean Δ: {all_pairs_cv['acc_diff'].mean():+.4f}")
print(f"  Std Δ: {all_pairs_cv['acc_diff'].std():.4f}")
print(f"  Pairs with decreased discriminability (Δ > 0.02): {(all_pairs_cv['acc_diff'] > 0.02).sum():,}")
print(f"  Pairs with increased discriminability (Δ < -0.02): {(all_pairs_cv['acc_diff'] < -0.02).sum():,}")

TASK-INDUCED PAIRWISE CHANGES

Overall pairwise change (CV - Task):
  Mean Δ: +0.0383
  Std Δ: 0.0499
  Pairs with decreased discriminability (Δ > 0.02): 16,538
  Pairs with increased discriminability (Δ < -0.02): 3,049


In [51]:
# =============================================================================
# HYPOTHESIS H⁴: PERMUTATION TEST FOR TASK EFFECT
# =============================================================================

print("\n" + "="*100)
print("HYPOTHESIS H⁴: TASK-INDUCED PAIRWISE REORGANIZATION")
print("="*100)
print("\nH₀: Pairwise discriminability unchanged between Rest and Task")
print("H₁: Task systematically alters pairwise discriminability\n")

def permutation_test_pairwise(pw_cv, pw_task, n_perm=10000):
    """Permutation test for pairwise accuracy difference."""
    # Get upper triangles
    idx = np.triu_indices(pw_cv.shape[0], k=1)
    cv_vals = pw_cv[idx]
    task_vals = pw_task[idx]
    
    # Observed statistic: mean absolute difference
    observed = np.mean(np.abs(cv_vals - task_vals))
    
    # Permutation
    null_dist = []
    combined = np.stack([cv_vals, task_vals])
    
    for _ in range(n_perm):
        # Randomly swap CV/Task labels for each pair
        swaps = np.random.randint(0, 2, size=len(cv_vals))
        perm_cv = np.where(swaps == 0, combined[0], combined[1])
        perm_task = np.where(swaps == 0, combined[1], combined[0])
        perm_diff = np.mean(np.abs(perm_cv - perm_task))
        null_dist.append(perm_diff)
    
    p_value = np.mean(np.array(null_dist) >= observed)
    
    return observed, p_value, np.array(null_dist)

print("Running permutation test (10,000 permutations)...")
obs, p_val, null = permutation_test_pairwise(pw_full_ovo_cv, pw_full_ovo_task)

print(f"\nObserved mean |Δ|: {obs:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"\nConclusion: {'Reject H₀ — Task alters pairwise patterns' if p_val < 0.05 else 'Fail to reject H₀'}")


HYPOTHESIS H⁴: TASK-INDUCED PAIRWISE REORGANIZATION

H₀: Pairwise discriminability unchanged between Rest and Task
H₁: Task systematically alters pairwise discriminability

Running permutation test (10,000 permutations)...

Observed mean |Δ|: 0.0491
p-value: 1.0000

Conclusion: Fail to reject H₀


In [52]:
# =============================================================================
# PAIRS WITH LARGEST TASK EFFECT
# =============================================================================

print("\n" + "-"*120)
print("PAIRS WITH LARGEST TASK-INDUCED DECREASE (became more similar during task):")
print("-"*120)

largest_decrease = all_pairs_cv.nlargest(20, 'acc_diff')
print(f"{'Region A':<30} {'Region B':<30} {'CV':>7} {'Task':>7} {'Δ':>8}")
for _, row in largest_decrease.iterrows():
    print(f"{row['name_i'][:29]:<30} {row['name_j'][:29]:<30} {row['acc_cv']:>6.1%} {row['acc_task']:>6.1%} {row['acc_diff']:>+7.1%}")

print("\n" + "-"*120)
print("PAIRS WITH LARGEST TASK-INDUCED INCREASE (became more distinct during task):")
print("-"*120)

largest_increase = all_pairs_cv.nsmallest(20, 'acc_diff')
for _, row in largest_increase.iterrows():
    print(f"{row['name_i'][:29]:<30} {row['name_j'][:29]:<30} {row['acc_cv']:>6.1%} {row['acc_task']:>6.1%} {row['acc_diff']:>+7.1%}")


------------------------------------------------------------------------------------------------------------------------
PAIRS WITH LARGEST TASK-INDUCED DECREASE (became more similar during task):
------------------------------------------------------------------------------------------------------------------------
Region A                       Region B                            CV    Task        Δ
RH_SalVentAttnA_PrC_1          RH_ContA_PFCl_1                 83.0%  55.8%  +27.3%
RH_SalVentAttnA_PrC_1          aGP-rh                          63.6%  37.5%  +26.1%
RH_ContA_PFCl_1                aGP-rh                          68.1%  42.2%  +25.8%
RH_SalVentAttnA_PrC_1          pPUT-lh                         83.3%  58.8%  +24.5%
RH_ContA_PFCl_1                pPUT-lh                         87.7%  63.5%  +24.2%
RH_SalVentAttnA_PrC_1          pHIP-lh                         77.9%  54.2%  +23.7%
RH_ContA_PFCl_1                pHIP-lh                         82.4%  59.0%  +23.4%
aGP-rh

**Global Trend**
Mean Δ = +3.8% → task slightly reduces discriminability overall, but effect not significant (p = 1.0).

**Local Effects**
Some region pairs change drastically (up to ±27%).
- Convergence → regions co-activated, harder to distinguish (control, subcortical hubs)
- Divergence → regions segregate, easier to distinguish (salience vs default mode)

**Interpretation**
- Task selectively reorganizes specific pairs of regions, but there is no systematic global change in all pairwise relationships.
- This aligns with the idea that task-induced functional reorganization is targeted, not uniform across the brain.


**Most changes are local, not global → task reshapes connectivity in specific networks, not the whole brain**

In [76]:
import numpy as np
import pandas as pd
import plotly.express as px

# =============================================================================
# 1. DATA CATEGORIZATION & PERCENTAGE CALCULATION
# =============================================================================

ACC_THRESHOLD = 0.05
EFFECT_THRESHOLD = 1.2 

def categorize_effect(row):
    if row['effect_score'] >= EFFECT_THRESHOLD:
        if row['acc_diff'] >= ACC_THRESHOLD: 
            return 'Significant Accuracy DROP'  # Regions became more similar
        elif row['acc_diff'] <= -ACC_THRESHOLD: 
            return 'Significant Accuracy GAIN'  # Regions became more distinct
    return 'Stable / Minor Change'

all_pairs_cv['impact_status'] = all_pairs_cv.apply(categorize_effect, axis=1)

# Calculate percentages for the summary box
def calc_stats(ptype):
    subset = all_pairs_cv[all_pairs_cv['pair_type'] == ptype]
    total = len(subset)
    gain = (subset['impact_status'] == 'Significant Accuracy GAIN').sum() / total * 100
    drop = (subset['impact_status'] == 'Significant Accuracy DROP').sum() / total * 100
    return f"<b>{ptype}:</b> {gain:.1f}% Gain | {drop:.1f}% Drop"

summary_stats = f"{calc_stats('Within')}<br>{calc_stats('Cross')}"

sample_pairs = all_pairs_cv.sample(n=min(5000, len(all_pairs_cv)), random_state=42)

# =============================================================================
# 2. CREATE THE PLOT
# =============================================================================

fig = px.scatter(
    sample_pairs, x='acc_diff', y='effect_score',
    color='impact_status',
    symbol='pair_type',
    color_discrete_map={
        'Significant Accuracy GAIN': '#00cc96', # Green (Left)
        'Significant Accuracy DROP': '#ef553b', # Red (Right)
        'Stable / Minor Change': '#dfe6e9'    # Grey (Center)
    },
    opacity=0.6,
    marginal_x="histogram", 
    hover_data={'acc_diff': ':.1%', 'effect_score': ':.2f', 'name_i': True, 'name_j': True},
    title='<b>Pairwise Task Effects: Within vs. Cross Analysis</b>',
    template='plotly_white',
    height=750, width=1100
)

# =============================================================================
# 3. DIRECT DATA INTERPRETATION (Minimal Labels)
# =============================================================================

# Summary Stats Box (Top Center-Left)
fig.add_annotation(
    x=0.02, y=0.98, xref="paper", yref="paper",
    text=f"<b>Performance Summary</b><br>{summary_stats}",
    showarrow=False, align="left",
    bgcolor="white", bordercolor="black", borderwidth=1, borderpad=8
)

# Strategic Quadrant Labels
fig.add_annotation(x=-0.12, y=1.65, text="<b>ACCURACY IMPROVED</b>", showarrow=False, font=dict(color="#00cc96", size=13))
fig.add_annotation(x=0.25, y=1.65, text="<b>ACCURACY DROPPED</b>", showarrow=False, font=dict(color="#ef553b", size=13))

# Threshold Guidelines
fig.add_vline(x=ACC_THRESHOLD, line_dash='dot', line_color='#b2bec3')
fig.add_vline(x=-ACC_THRESHOLD, line_dash='dot', line_color='#b2bec3')
fig.add_hline(y=EFFECT_THRESHOLD, line_dash='dash', line_color='#636e72', annotation_text="Significance Cutoff")
fig.add_vline(x=0, line_color='black', line_width=1.5)

# =============================================================================
# 4. FINAL LAYOUT
# =============================================================================

fig.update_layout(
    xaxis_title='<b>Magnitude of Change</b> (CV Baseline - Task Performance)',
    yaxis_title='<b>Effect Score</b> (Impact / Variance)',
    xaxis=dict(tickformat='.0%', range=[-0.20, 0.40]),
    yaxis=dict(range=[-0.1, 1.8]),
    legend=dict(title_text='<b>Impact Status</b>', orientation="h", y=-0.15),
    margin=dict(t=120)
)

fig.show()

---
## 9. Confusion Asymmetry Analysis (H⁶)

In [54]:
# =============================================================================
# CONFUSION ASYMMETRY CALCULATION
# =============================================================================

def compute_confusion_asymmetry(y_true, y_pred, n_regions):
    """
    Compute directional confusion counts.
    
    For each pair (i, j):
    - FP_i_to_j: True label is i, predicted as j
    - FP_j_to_i: True label is j, predicted as i
    """
    cm = confusion_matrix(y_true, y_pred, labels=range(n_regions))
    
    asymmetry_data = []
    
    for i in range(n_regions):
        for j in range(i+1, n_regions):
            fp_i_to_j = cm[i, j]  # True i, pred j
            fp_j_to_i = cm[j, i]  # True j, pred i
            total = fp_i_to_j + fp_j_to_i
            
            if total > 0:
                asymmetry = (fp_i_to_j - fp_j_to_i) / total
            else:
                asymmetry = 0
            
            asymmetry_data.append({
                'region_i': i,
                'region_j': j,
                'fp_i_to_j': fp_i_to_j,
                'fp_j_to_i': fp_j_to_i,
                'total_confusion': total,
                'asymmetry': asymmetry,
                'abs_asymmetry': abs(asymmetry),
                'dominant_direction': f'{i}→{j}' if fp_i_to_j > fp_j_to_i else (f'{j}→{i}' if fp_j_to_i > fp_i_to_j else 'Equal')
            })
    
    return pd.DataFrame(asymmetry_data)

asymmetry_cv = compute_confusion_asymmetry(
    full_ovo_cv['true_labels'], full_ovo_cv['predictions'], N_REGIONS_FULL)

# Add region names
asymmetry_cv['name_i'] = asymmetry_cv['region_i'].map(lambda x: region_info.iloc[x]['region_name'])
asymmetry_cv['name_j'] = asymmetry_cv['region_j'].map(lambda x: region_info.iloc[x]['region_name'])

print("✓ Confusion asymmetry computed")

✓ Confusion asymmetry computed


In [55]:
# =============================================================================
# HYPOTHESIS H⁶: CONFUSION ASYMMETRY
# =============================================================================

print("="*100)
print("HYPOTHESIS H⁶: CONFUSION ASYMMETRY")
print("="*100)
print("\nH₀: Confusion is symmetric P(A→B) = P(B→A)")
print("H₁: Certain region pairs show asymmetric confusion\n")

# Filter pairs with at least some confusion
confused_pairs = asymmetry_cv[asymmetry_cv['total_confusion'] >= 5].copy()

# Binomial test for each pair
def binomial_asymmetry_test(row):
    if row['total_confusion'] == 0:
        return 1.0
    # Test if observed proportion differs from 0.5
    k = max(row['fp_i_to_j'], row['fp_j_to_i'])
    n = row['total_confusion']
    # Two-tailed binomial test
    p = 2 * min(stats.binom.cdf(n - k, n, 0.5), 1 - stats.binom.cdf(k - 1, n, 0.5))
    return min(p, 1.0)

confused_pairs['p_value'] = confused_pairs.apply(binomial_asymmetry_test, axis=1)

# FDR correction
_, confused_pairs['p_adjusted'], _, _ = multipletests(
    confused_pairs['p_value'], method='fdr_bh')

confused_pairs['significant'] = confused_pairs['p_adjusted'] < 0.05

n_sig = confused_pairs['significant'].sum()
print(f"Pairs with ≥5 confusions: {len(confused_pairs):,}")
print(f"Pairs with significant asymmetry (FDR < 0.05): {n_sig:,} ({n_sig/len(confused_pairs)*100:.1f}%)")

# Show top asymmetric pairs
print("\n" + "-"*120)
print("TOP 20 MOST ASYMMETRIC PAIRS:")
print("-"*120)

top_asym = confused_pairs.nlargest(20, 'abs_asymmetry')
print(f"{'Region A':<30} {'Region B':<30} {'A→B':>6} {'B→A':>6} {'Asym':>8} {'p_adj':>10}")
for _, row in top_asym.iterrows():
    sig = '*' if row['significant'] else ''
    print(f"{row['name_i'][:29]:<30} {row['name_j'][:29]:<30} {row['fp_i_to_j']:>6} {row['fp_j_to_i']:>6} "
          f"{row['asymmetry']:>+7.2f} {row['p_adjusted']:>9.2e} {sig}")

HYPOTHESIS H⁶: CONFUSION ASYMMETRY

H₀: Confusion is symmetric P(A→B) = P(B→A)
H₁: Certain region pairs show asymmetric confusion

Pairs with ≥5 confusions: 149
Pairs with significant asymmetry (FDR < 0.05): 0 (0.0%)

------------------------------------------------------------------------------------------------------------------------
TOP 20 MOST ASYMMETRIC PAIRS:
------------------------------------------------------------------------------------------------------------------------
Region A                       Region B                          A→B    B→A     Asym      p_adj
LH_DorsAttnB_FEF_1             RH_SalVentAttnA_PrC_1               6      0   +1.00  1.00e+00 
LH_DefaultB_PFCv_2             NAc-shell-rh                        5      0   +1.00  1.00e+00 
LH_TempPar_2                   RH_TempPar_2                        0      5   -1.00  1.00e+00 
RH_DefaultA_PFCm_1             RH_DefaultA_PFCm_2                  5      0   +1.00  1.00e+00 
lAMY-rh                        aHI

---
## 10. Integration with Error-as-Signal Framework

In [56]:
# =============================================================================
# SYNTHESIS: THREE-LEVEL ERROR INTERPRETATION
# =============================================================================

print("="*140)
print("INTEGRATION: THREE LEVELS OF ERROR INTERPRETATION")
print("="*140)

print("""
┌─────────────┬───────────────────────────────────┬───────────────────────────────────────────────────────────┐
│   Level     │           Question                │                    Error Meaning                          │
├─────────────┼───────────────────────────────────┼───────────────────────────────────────────────────────────┤
│ Multinomial │ "Which region?"                   │ Global misclassification — cannot decompose cause         │
├─────────────┼───────────────────────────────────┼───────────────────────────────────────────────────────────┤
│ OvR         │ "This region or not?"             │ Weak fingerprint (low sens) OR strong competitor (low spec) │
├─────────────┼───────────────────────────────────┼───────────────────────────────────────────────────────────┤
│ OvO         │ "This vs. that region?"           │ Direct functional similarity — pairwise confusion         │
└─────────────┴───────────────────────────────────┴───────────────────────────────────────────────────────────┘

KEY INSIGHT: OvO transforms classification errors into a SIMILARITY METRIC.

If regions A and B are frequently confused:
  → Their connectivity fingerprints are similar
  → They may be functionally coupled
  → Task-induced changes in confusion reflect changes in functional coupling
""")

INTEGRATION: THREE LEVELS OF ERROR INTERPRETATION

┌─────────────┬───────────────────────────────────┬───────────────────────────────────────────────────────────┐
│   Level     │           Question                │                    Error Meaning                          │
├─────────────┼───────────────────────────────────┼───────────────────────────────────────────────────────────┤
│ Multinomial │ "Which region?"                   │ Global misclassification — cannot decompose cause         │
├─────────────┼───────────────────────────────────┼───────────────────────────────────────────────────────────┤
│ OvR         │ "This region or not?"             │ Weak fingerprint (low sens) OR strong competitor (low spec) │
├─────────────┼───────────────────────────────────┼───────────────────────────────────────────────────────────┤
│ OvO         │ "This vs. that region?"           │ Direct functional similarity — pairwise confusion         │
└─────────────┴───────────────────────────────────┴

In [57]:
# =============================================================================
# PAIRWISE SIMILARITY AS FUNCTIONAL COUPLING
# =============================================================================

# Convert pairwise accuracy to similarity (1 - accuracy = confusion rate)
pw_similarity_cv = 1 - pw_full_ovo_cv
pw_similarity_task = 1 - pw_full_ovo_task

# Network-level mean similarity (confusion rate)
net_sim_cv = 1 - net_D_cv
net_sim_task = 1 - net_D_task

print("="*100)
print("PAIRWISE CONFUSION AS FUNCTIONAL SIMILARITY")
print("="*100)

print("\nNetwork-Pair Mean Confusion Rate (= 1 - Accuracy):")
print("(Higher = more similar/confusable)\n")

# Create ranked list
net_pairs_sim = []
for i in range(N_NETWORKS):
    for j in range(i, N_NETWORKS):
        net_pairs_sim.append({
            'Network Pair': f"{NETWORKS[i][:8]} × {NETWORKS[j][:8]}",
            'CV Confusion': net_sim_cv[i, j],
            'Task Confusion': net_sim_task[i, j],
            'Δ': net_sim_task[i, j] - net_sim_cv[i, j]
        })

net_pairs_sim_df = pd.DataFrame(net_pairs_sim).sort_values('CV Confusion', ascending=False)

print(f"{'Network Pair':<25} {'CV Conf':>10} {'Task Conf':>11} {'Δ':>10}")
print("-"*60)
for _, row in net_pairs_sim_df.head(15).iterrows():
    print(f"{row['Network Pair']:<25} {row['CV Confusion']:>9.1%} {row['Task Confusion']:>10.1%} {row['Δ']:>+9.1%}")

PAIRWISE CONFUSION AS FUNCTIONAL SIMILARITY

Network-Pair Mean Confusion Rate (= 1 - Accuracy):
(Higher = more similar/confusable)

Network Pair                 CV Conf   Task Conf          Δ
------------------------------------------------------------
Subcorti × Subcorti           22.5%      31.9%     +9.4%
Limbic × Subcorti             21.9%      26.6%     +4.6%
Limbic × Limbic               21.4%      21.3%     -0.1%
Salience × Subcorti           18.0%      24.8%     +6.8%
Salience × Limbic             17.5%      19.4%     +2.0%
Default × Subcorti            17.4%      23.8%     +6.3%
Control × Subcorti            17.0%      23.2%     +6.3%
Dorsal A × Subcorti           16.9%      22.0%     +5.1%
Limbic × Default              16.9%      18.5%     +1.6%
Limbic × Control              16.4%      17.9%     +1.5%
Dorsal A × Limbic             16.4%      16.6%     +0.3%
Somatomo × Subcorti           15.9%      22.4%     +6.5%
Somatomo × Limbic             15.4%      17.1%     +1.7%
Visual

Observation
- Highest Overall Similarity,Subcorti × Subcorti (31.9%),These regions are the hardest for the model to tell apart during the task.

- Largest Task-Induced Shift,Subcorti × Subcorti (+9.4%),The task fundamentally changes how these regions communicate compared to baseline.

- Most Stable Relationship,Limbic × Limbic (−0.1%),Task demands do not alter the functional distinction of Limbic regions.

- Cortical-Subcortical Link,Visual/Somatomo × Subcorti,Significant increases (+6.5% to +6.6%) show motor/visual systems syncing with deeper brain structures.

In [97]:
import plotly.graph_objects as go
import pandas as pd

# 1. Prepare and sort data from your existing net_pairs_sim_df
# Sorting by CV Confusion to create a clear visual hierarchy
plot_df = net_pairs_sim_df.head(15).sort_values('CV Confusion', ascending=True)

# 2. Initialize the Figure
fig = go.Figure()

# 3. Add Dumbbell Components
# Add the connecting lines
for i, row in plot_df.iterrows():
    fig.add_shape(
        type='line',
        x0=row['CV Confusion'], x1=row['Task Confusion'],
        y0=row['Network Pair'], y1=row['Network Pair'],
        line=dict(color='#D5DBDB', width=1.5),
        layer='below'
    )

# Add Baseline (Rest) markers
fig.add_trace(go.Scatter(
    x=plot_df['CV Confusion'],
    y=plot_df['Network Pair'],
    mode='markers',
    name='Baseline (Rest)',
    marker=dict(color='#34495E', size=7, line=dict(color='white', width=0.5))
))

# Add Task State markers
fig.add_trace(go.Scatter(
    x=plot_df['Task Confusion'],
    y=plot_df['Network Pair'],
    mode='markers',
    name='Task State',
    marker=dict(color='#E74C3C', size=7, line=dict(color='white', width=0.5))
))

# 4. Add Delta Annotations (Fixed $ and increased font size)
annotations = []
for i, row in plot_df.iterrows():
    delta_val = row['Δ']
    color = '#C0392B' if delta_val > 0 else '#2E86C1'
    
    annotations.append(dict(
        x=max(row['CV Confusion'], row['Task Confusion']) + 0.006,
        y=row['Network Pair'],
        text=f"<b>{delta_val:+.1%}</b>", # Removed stray $
        showarrow=False,
        xanchor='left',
        font=dict(size=11, color=color) # Increased font size to 11
    ))

# 5. Professional Layout Adjustments
fig.update_layout(
    title=dict(
        text='<b>Functional Coupling Shifts: Rest vs. Task</b>',
        font=dict(size=13, color='#2C3E50'),
        x=0.5,
        y=0.97
    ),
    width=650,   
    height=480,  
    margin=dict(l=150, r=80, t=60, b=50),
    xaxis=dict(
        title=dict(
            text='Mean Confusion Rate',
            font=dict(size=11, color='#2C3E50')
        ),
        tickfont=dict(size=10),
        tickformat='.0%',
        gridcolor='#F2F3F4',
        zeroline=False,
        range=[plot_df['CV Confusion'].min() - 0.03, plot_df['Task Confusion'].max() + 0.08]
    ),
    yaxis=dict(
        tickfont=dict(size=10),
        gridcolor='#F2F3F4',
        automargin=True
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        font=dict(size=10)
    ),
    plot_bgcolor='white',
    annotations=annotations
)

fig.show()

---
## 11. Hypothesis Testing Summary

In [58]:
# =============================================================================
# HYPOTHESIS TESTING SUMMARY
# =============================================================================

print("="*160)
print("HYPOTHESIS TESTING SUMMARY")
print("="*160)

print("""
┌────────┬────────────────────────────────────────────────────────────┬─────────────────────┬──────────────┬───────────────┐
│   ID   │                        Hypothesis                           │        Test         │   p-value    │    Result     │
├────────┼────────────────────────────────────────────────────────────┼─────────────────────┼──────────────┼───────────────┤""")

# H1: Three-way comparison
print(f"│   H¹   │ Multi = OvR = OvO accuracy                                 │ Cochran's Q         │ {q_result['p_value']:>10.2e} │ {'Reject H₀' if q_result['p_value'] < 0.05 else 'Fail to reject':<13} │")

# H2: Within vs Cross
print(f"│   H²   │ Within-network = Cross-network pairwise accuracy           │ Mann-Whitney U      │ {p_one:>10.2e} │ {'Reject H₀' if p_one < 0.05 else 'Fail to reject':<13} │")

# H3: Homologous pairs
print(f"│   H³   │ Homologous = Non-homologous discriminability                │ Mann-Whitney U      │ {u_p:>10.2e} │ {'Reject H₀' if u_p < 0.05 else 'Fail to reject':<13} │")

# H4: Task effect
print(f"│   H⁴   │ Pairwise discriminability same Rest/Task                    │ Permutation         │ {p_val:>10.4f} │ {'Reject H₀' if p_val < 0.05 else 'Fail to reject':<13} │")

# H5: Network pairs differ
print(f"│   H⁵   │ All network pairs equally discriminable                     │ Kruskal-Wallis      │ {h_p:>10.2e} │ {'Reject H₀' if h_p < 0.05 else 'Fail to reject':<13} │")

# H6: Asymmetry
print(f"│   H⁶   │ Confusion symmetric P(A→B) = P(B→A)                         │ Binomial + FDR      │      —       │ {n_sig} pairs sig  │")

print("└────────┴────────────────────────────────────────────────────────────┴─────────────────────┴──────────────┴───────────────┘")

HYPOTHESIS TESTING SUMMARY

┌────────┬────────────────────────────────────────────────────────────┬─────────────────────┬──────────────┬───────────────┐
│   ID   │                        Hypothesis                           │        Test         │   p-value    │    Result     │
├────────┼────────────────────────────────────────────────────────────┼─────────────────────┼──────────────┼───────────────┤
│   H¹   │ Multi = OvR = OvO accuracy                                 │ Cochran's Q         │   0.00e+00 │ Reject H₀     │
│   H²   │ Within-network = Cross-network pairwise accuracy           │ Mann-Whitney U      │   1.00e+00 │ Fail to reject │
│   H³   │ Homologous = Non-homologous discriminability                │ Mann-Whitney U      │   1.78e-03 │ Reject H₀     │
│   H⁴   │ Pairwise discriminability same Rest/Task                    │ Permutation         │     1.0000 │ Fail to reject │
│   H⁵   │ All network pairs equally discriminable                     │ Kruskal-Wallis      │   0.0

---
## 12. Conclusions

In [59]:
# =============================================================================
# COMPREHENSIVE SUMMARY
# =============================================================================

print("="*140)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("="*140)

print(f"""
1. THREE-WAY CLASSIFICATION COMPARISON
   ─────────────────────────────────────────────────────────────────────────
   Full Model Accuracy (CV):
   ├── Multinomial:  {metrics_all['full_multi_cv']['accuracy']:.2%}
   ├── OvR:          {metrics_all['full_ovr_cv']['accuracy']:.2%}
   └── OvO:          {metrics_all['full_ovo_cv']['accuracy']:.2%}
   
   Cochran's Q p-value: {q_result['p_value']:.2e}

2. PAIRWISE DISCRIMINABILITY
   ─────────────────────────────────────────────────────────────────────────
   Total pairs analyzed: {N_PAIRS_FULL:,}
   Mean pairwise accuracy (CV): {stats_list[0]['mean']:.2%}
   Pairs below 90% accuracy: {stats_list[0]['below_90']:,}
   Pairs below 80% accuracy: {stats_list[0]['below_80']:,}

3. NETWORK EFFECTS
   ─────────────────────────────────────────────────────────────────────────
   Within-network pairs: Mean = {np.mean(within):.4f}
   Cross-network pairs:  Mean = {np.mean(cross):.4f}
   Difference: {np.mean(cross) - np.mean(within):+.4f} (Cross MORE discriminable)

4. HOMOLOGOUS PAIRS
   ─────────────────────────────────────────────────────────────────────────
   Identified: {len(homologous_pairs)} pairs
   Mean accuracy: {homologous_pairs['acc_cv'].mean():.4f}
   vs Non-homologous: {np.mean(non_homo_acc):.4f}

5. TASK-INDUCED CHANGES
   ─────────────────────────────────────────────────────────────────────────
   Mean pairwise change (CV - Task): {all_pairs_cv['acc_diff'].mean():+.4f}
   Pairs with >2% decrease: {(all_pairs_cv['acc_diff'] > 0.02).sum():,}
   Pairs with >2% increase: {(all_pairs_cv['acc_diff'] < -0.02).sum():,}

6. CONFUSION ASYMMETRY
   ─────────────────────────────────────────────────────────────────────────
   Pairs with significant asymmetry: {n_sig:,}

KEY CONCLUSIONS
─────────────────────────────────────────────────────────────────────────
• OvO provides unique insights into pairwise functional similarity
• Cross-network pairs are more discriminable than within-network pairs
• Homologous inter-hemispheric pairs show distinct patterns
• Task engagement alters pairwise discriminability systematically
• The "error-as-signal" framework is validated: confusion ∝ functional similarity
""")

print("="*140)

COMPREHENSIVE ANALYSIS SUMMARY

1. THREE-WAY CLASSIFICATION COMPARISON
   ─────────────────────────────────────────────────────────────────────────
   Full Model Accuracy (CV):
   ├── Multinomial:  92.41%
   ├── OvR:          93.17%
   └── OvO:          86.88%

   Cochran's Q p-value: 0.00e+00

2. PAIRWISE DISCRIMINABILITY
   ─────────────────────────────────────────────────────────────────────────
   Total pairs analyzed: 26,796
   Mean pairwise accuracy (CV): 86.88%
   Pairs below 90% accuracy: 18,576
   Pairs below 80% accuracy: 2,682

3. NETWORK EFFECTS
   ─────────────────────────────────────────────────────────────────────────
   Within-network pairs: Mean = 0.8711
   Cross-network pairs:  Mean = 0.8686
   Difference: -0.0025 (Cross MORE discriminable)

4. HOMOLOGOUS PAIRS
   ─────────────────────────────────────────────────────────────────────────
   Identified: 78 pairs
   Mean accuracy: 0.8870
   vs Non-homologous: 0.8687

5. TASK-INDUCED CHANGES
   ───────────────────────────

In [60]:
# =============================================================================
# FINAL THESIS SUMMARY
# =============================================================================

print("="*120)
print("THESIS INTEGRATION: OvO AS A TOOL FOR FUNCTIONAL SIMILARITY MAPPING")
print("="*120)

print("""
One-vs-One classification uniquely contributes to brain connectivity analysis by:

1. TRANSFORMING ERRORS INTO SIMILARITY METRICS
   - Pairwise confusion rate directly quantifies functional similarity
   - Low pairwise accuracy = high similarity = potential functional coupling

2. REVEALING NETWORK BOUNDARY EFFECTS
   - Within-network pairs are harder to separate (more similar)
   - Cross-network pairs are more discriminable (more distinct)
   - This validates functional network organization

3. CHARACTERIZING TASK-INDUCED FUNCTIONAL REORGANIZATION
   - Changes in pairwise discriminability during task reflect:
     * Functional convergence (regions become more similar)
     * Functional differentiation (regions become more distinct)
   - Specific network pairs show systematic task effects

4. IDENTIFYING ASYMMETRIC CONFUSION PATTERNS
   - Some region pairs show directional confusion
   - May reflect hierarchical processing or dominant representations

5. COMPLEMENTING MULTINOMIAL AND OVR ANALYSES
   - Together, the three approaches provide:
     * Global performance (Multinomial)
     * Regional specificity (OvR)
     * Pairwise relationships (OvO)

The OvO framework establishes that classification "errors" are not failures,
but meaningful signals of functional brain organization.
""")

print("="*120)

THESIS INTEGRATION: OvO AS A TOOL FOR FUNCTIONAL SIMILARITY MAPPING

One-vs-One classification uniquely contributes to brain connectivity analysis by:

1. TRANSFORMING ERRORS INTO SIMILARITY METRICS
   - Pairwise confusion rate directly quantifies functional similarity
   - Low pairwise accuracy = high similarity = potential functional coupling

2. REVEALING NETWORK BOUNDARY EFFECTS
   - Within-network pairs are harder to separate (more similar)
   - Cross-network pairs are more discriminable (more distinct)
   - This validates functional network organization

3. CHARACTERIZING TASK-INDUCED FUNCTIONAL REORGANIZATION
   - Changes in pairwise discriminability during task reflect:
     * Functional convergence (regions become more similar)
     * Functional differentiation (regions become more distinct)
   - Specific network pairs show systematic task effects

4. IDENTIFYING ASYMMETRIC CONFUSION PATTERNS
   - Some region pairs show directional confusion
   - May reflect hierarchical proce